# In Class November 20, 2026

Working with the requests and the Omeka S API

In [1]:
import json
import requests

In [2]:
# establish API location/endpoints

siteUrl = 'http://jajohnst.si676.si.umich.edu/omekas_2026' # if you replicate this example, provide the URI for your site
endpoint = '/api'

Set up a process to get, but not publicize API credentials

In [3]:
def get_credentials(credential_file_path):
    '''Retrieve Omeka S API credentials from another file. 
    That file must be a JSON file.'''

    with open(credential_file_path, 'r') as credentials:
        keys = json.load(credentials)
    
    return keys['key_identity'], keys['key_credential']

Nota bene: the following cell was added and not used during class; this cell shows how to run the credentials function (`get_credentials()`), which outputs a tuple; then those values are assigned to two variables, one called `key_identity` and the other `key_credential`:

In [4]:
key_identity, key_credential = get_credentials('../omeka-credentials.json')

Set up the API request

In [5]:
# set up API call
action = '/items'

r_url = siteUrl + endpoint + action

print(r_url)

http://jajohnst.si676.si.umich.edu/omekas_2026/api/items


In [6]:
parameters = {
    'key_credential': key_credential,
    'key_identity':   key_identity,
    'pretty_print':   1,
    'format':         'jsonld'
}

In [7]:
r = requests.get(r_url, params=parameters)

print(r.status_code)

200


See if there are any results

In [8]:
len(r.json())

25

**\#\#\# Class demo of Nov 20 ended here \#\#\#**

### Below are additional directions to consider developing your work with the API

The response above suggests that there are 25 items listed in the Omeka S instance (all sites).
If you view the browse page of jajohnst's Omeka repo (<http://jajohnst.si676.si.umich.edu/omekas_2026/s/umsi-mumsi/item>) as of November 20, the list shows 35 total items, which suggests there is some difference between the items shown on the site and the items retrieved through the API.
An initial theory is that the API provides paginated results (defaulting to 25 results per response), whcih means any functions using this will need to work with that kind of structure, but whatever the cause this will require further investigation. That is, looking at the documentation (such as the [results per page global variable](https://omeka.org/s/docs/user-manual/admin/settings/#display))

### Look at the list

For example, you may want to look at the list. To do so, one way to start is to go through the items.
Recall that these responses use 

In [9]:
for item in r.json():
    print(item['o:id'],':', item['o:title'])

3 : For greater knowledge on more subjects use your library more often
5 : Curb service 10,000 current books - convenient, free, time saving : Chicago Public Library, Randolph St. corridor.
7 : For greater knowledge on more subjects use your library more often
9 : [Cossitt Library, Memphis, Tenn.]
11 : Public library in the southwestern section of Mississippi, in the piney woods
13 : Daytona Beach, Florida. Bethune-Cookman College. Students in the library reading room
15 : Libraries, Washington D.C.
17 : George Peabody Library, formerly the Library of the Peabody Institute of the City of Baltimore, is part of the Johns Hopkins Sheridan Libraries. Baltimore, Maryland
19 : Handley Library, Winchester, Virginia
21 : Public library building in the town of Antonito in Conejos County, Colorado, near the New Mexico border
23 : Modernist stairway and bookstacks inside the Allen County Public Library in Fort Wayne, Indiana
25 : Reading room for blind, Library of Congress, [Washington, D.C.]
27 

To see the response, use the `json` library; this may be useful in identifying a pagination variable:

In [10]:
print(json.dumps(r.json(), indent=2))

[
  {
    "@context": "http://jajohnst.si676.si.umich.edu/omekas_2026/api-context",
    "@id": "http://jajohnst.si676.si.umich.edu/omekas_2026/api/items/3",
    "@type": [
      "o:Item",
      "dcterms:MediaType"
    ],
    "o:id": 3,
    "o:is_public": true,
    "o:owner": {
      "@id": "http://jajohnst.si676.si.umich.edu/omekas_2026/api/users/1",
      "o:id": 1
    },
    "o:resource_class": {
      "@id": "http://jajohnst.si676.si.umich.edu/omekas_2026/api/resource_classes/11",
      "o:id": 11
    },
    "o:resource_template": {
      "@id": "http://jajohnst.si676.si.umich.edu/omekas_2026/api/resource_templates/2",
      "o:id": 2
    },
    "o:thumbnail": null,
    "o:title": "For greater knowledge on more subjects use your library more often",
    "thumbnail_display_urls": {
      "large": "http://jajohnst.si676.si.umich.edu/omekas_2026/files/large/b3b75b89bb77dfafdbfd32f17d5450380a355a98.jpg",
      "medium": "http://jajohnst.si676.si.umich.edu/omekas_2026/files/medium/b3b75b

On a closer look, the [Omeka API documentation suggests](https://omeka.org/s/docs/developer/api/rest_api/) that pagination is indicated in the response headers. Note that these headers will contain the credentials, since they are passed as part of the URL request to the REST API. Therefore, the response is not shown below:

In [ ]:
r.headers

It's a dictionary, and you can list the keys:

In [12]:
for header in r.headers:
    print(header)


Server
Date
Content-Type
Transfer-Encoding
Connection
Link
Omeka-S-Total-Results
Omeka-S-Version


And get specific values:

In [13]:
r.headers.get('Omeka-S-Total-Results')

'35'

There's the total (and it matches the expected number).
It turns out information about pagination is in the `Link` header, which contains information about how to navigate (like previous, next links, etc). Find it by treating it as a subdictionary. Note, however, that the links will contain the credentials, so the response is not displayed below:

In [ ]:
r.links